# 🚦 Module 05: Concurrency Controls — Locks, Semaphores & Queues

Even though Python `asyncio` runs on a single thread, **race conditions can still corrupt your data**!
Why? Because when a coroutine hits an `await`, the event loop switches to another coroutine that modifies the exact same shared state.

### What We Will Master:
1. **Race Conditions in Single-Threaded Async** (Proof with reproducible bug).
2. **`asyncio.Lock`**: Protecting critical shared resources.
3. **`asyncio.Semaphore`**: Rate limiting third-party APIs (e.g. OpenAI / Stripe).
4. **`asyncio.BoundedSemaphore`**: Catching accidental over-releases.
5. **`asyncio.Event`**: Signaling and coordinating worker readiness.
6. **`asyncio.Queue`**: Multi-producer, multi-consumer pipelines with poison pills (`None`).
7. **`asyncio.PriorityQueue`**: Processing VIP customer requests first.

In [1]:
import asyncio

# 1. Unsafe Inventory Decrement
stock = 10
async def buy_item_unsafe(user_id: int):
    global stock
    current_stock = stock
    await asyncio.sleep(0.01) # Event loop switches here to 9 other users!
    stock = current_stock - 1

await asyncio.gather(*(buy_item_unsafe(i) for i in range(10)))
print("❌ UNPROTECTED INVENTORY (BUG):")
print("Expected remaining stock: 0")
print(f"Actual corrupted stock: {stock} (Over-sold to multiple users! Race condition destroyed inventory!)")
print("-" * 50)

# 2. Safe Inventory Decrement with Lock
safe_stock = 10
inventory_lock = asyncio.Lock()

async def buy_item_safe(user_id: int):
    global safe_stock
    async with inventory_lock:
        current_stock = safe_stock
        await asyncio.sleep(0.01)
        safe_stock = current_stock - 1

await asyncio.gather(*(buy_item_safe(i) for i in range(10)))
print("✅ PROTECTED INVENTORY (WITH ASYNCIO.LOCK):")
print("Expected remaining stock: 0")
print(f"Actual protected stock: {safe_stock} (Transactions accurately synchronized!)")

❌ UNPROTECTED INVENTORY (BUG):
Expected remaining stock: 0
Actual corrupted stock: 8 (Over-sold to multiple users! Race condition destroyed inventory!)
--------------------------------------------------
✅ PROTECTED INVENTORY (WITH ASYNCIO.LOCK):
Expected remaining stock: 0
Actual protected stock: 0 (Transactions accurately synchronized!)


---
## 2. `asyncio.Event`: Coordinating Dependent Workers

An `Event` acts as a green traffic light. Multiple workers can call `await event.wait()` to pause until another worker calls `event.set()`.

In [2]:
cache_ready_event = asyncio.Event()

async def worker(worker_id: int):
    print(f"Worker {worker_id} waiting for Redis Cache to be warmed up...")
    await cache_ready_event.wait() # Pauses here until event is set!
    print(f"🚀 Worker {worker_id} received green light! Processing requests instantly from cache.")

async def cache_warmer():
    print("🔄 Cache warmer pre-fetching 5,000 product keys into RAM...")
    await asyncio.sleep(0.4) # Warmup time
    print("🟢 Cache warmer finished! Firing event.set()...")
    cache_ready_event.set() # Unblocks all waiting workers!

await asyncio.gather(worker(1), worker(2), cache_warmer())

Worker 1 waiting for Redis Cache to be warmed up...
Worker 2 waiting for Redis Cache to be warmed up...
🔄 Cache warmer pre-fetching 5,000 product keys into RAM...
🟢 Cache warmer finished! Firing event.set()...
🚀 Worker 1 received green light! Processing requests instantly from cache.
🚀 Worker 2 received green light! Processing requests instantly from cache.


---
## 3. `asyncio.PriorityQueue`: VIP Customer Routing

In production APIs, free tier users shouldn't block enterprise paid users. 
Items are stored as `(priority, data)`. Lower numbers are processed first!

In [3]:
pq = asyncio.PriorityQueue()

# Priority 1 = VIP Enterprise, Priority 10 = Free Tier
await pq.put((10, "Free tier user 1"))
print("Queued: Free tier user 1 (Priority 10)")
await pq.put((10, "Free tier user 2"))
print("Queued: Free tier user 2 (Priority 10)")
await pq.put((1, "Enterprise VIP user 99"))
print("Queued: Enterprise VIP user 99 (Priority 1)")

while not pq.empty():
    priority, item = await pq.get()
    suffix = " (Processed first despite arriving last!)" if priority == 1 else ""
    print(f"Consumer processing: {item}{suffix}")
    pq.task_done()

Queued: Free tier user 1 (Priority 10)
Queued: Free tier user 2 (Priority 10)
Queued: Enterprise VIP user 99 (Priority 1)
Consumer processing: Enterprise VIP user 99 (Processed first despite arriving last!)
Consumer processing: Free tier user 1
Consumer processing: Free tier user 2
